# Exercise 2.2: Branching, Tagging, and Time Travel

In this exercise, you'll learn how to use Apache Iceberg's versioning features:
- **Snapshots**: Every write creates a new version
- **Time Travel**: Query historical versions of your table
- **Tagging**: Mark important snapshots with memorable names
- **Rollback**: Recover from mistakes by reverting to previous states

## Learning Objectives
- Understand how snapshots track table history
- Use time travel to query past versions
- Create and use tags for important milestones
- Rollback to previous table states
- Understand snapshot isolation

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("BranchingAndTagging") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.timetravel")
print("Namespace 'timetravel' created!")

## Part 1: Create a Table and Build History

In [ ]:
# Create employees table
spark.sql("""
CREATE OR REPLACE TABLE polaris.timetravel.employees (
    employee_id INT,
    name STRING,
    department STRING,
    salary DOUBLE,
    hire_date DATE
)
USING iceberg
""")

print("Employees table created!")

In [ ]:
# Insert initial data - Snapshot 1
spark.sql("""
INSERT INTO polaris.timetravel.employees VALUES
    (1, 'Alice Johnson', 'Engineering', 95000, DATE '2020-01-15'),
    (2, 'Bob Smith', 'Sales', 75000, DATE '2019-03-22'),
    (3, 'Carol White', 'Engineering', 105000, DATE '2018-06-10')
""")

print("Initial employees inserted!")
spark.sql("SELECT * FROM polaris.timetravel.employees ORDER BY employee_id").show()

In [ ]:
# Add more employees - Snapshot 2
spark.sql("""
INSERT INTO polaris.timetravel.employees VALUES
    (4, 'David Brown', 'Marketing', 68000, DATE '2021-02-01'),
    (5, 'Eve Davis', 'Engineering', 92000, DATE '2020-11-15')
""")

print("More employees added!")
spark.sql("SELECT * FROM polaris.timetravel.employees ORDER BY employee_id").show()

In [ ]:
# Give everyone a 10% raise - Snapshot 3
spark.sql("""
UPDATE polaris.timetravel.employees
SET salary = salary * 1.10
""")

print("Salaries updated!")
spark.sql("SELECT employee_id, name, salary FROM polaris.timetravel.employees ORDER BY employee_id").show()

## Part 2: View Snapshot History

In [ ]:
# View all snapshots
print("Snapshot history:")
snapshots_df = spark.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation,
    summary
FROM polaris.timetravel.employees.snapshots
ORDER BY committed_at
""")
snapshots_df.show(truncate=False)

## Part 3: Time Travel - Query Historical Data

### Query Snapshot 1 (Original 3 Employees)

In [ ]:
# Get first snapshot ID
snapshot_1 = snapshots_df.collect()[0]['snapshot_id']
print(f"Snapshot 1 ID: {snapshot_1}")

# Query snapshot 1
print("\nSnapshot 1 (original 3 employees):")
spark.sql(f"""
SELECT * FROM polaris.timetravel.employees VERSION AS OF {snapshot_1}
ORDER BY employee_id
""").show()

### Query Snapshot 2 (5 Employees, Original Salaries)

In [ ]:
# Get second snapshot ID
snapshot_2 = snapshots_df.collect()[1]['snapshot_id']
print(f"Snapshot 2 ID: {snapshot_2}")

# Query snapshot 2
print("\nSnapshot 2 (5 employees, before raise):")
spark.sql(f"""
SELECT employee_id, name, salary FROM polaris.timetravel.employees VERSION AS OF {snapshot_2}
ORDER BY employee_id
""").show()

### Compare Current vs Historical

In [ ]:
# Current data (after 10% raise)
print("Current data (after 10% raise):")
current_df = spark.sql("""
SELECT employee_id, name, salary as current_salary
FROM polaris.timetravel.employees
ORDER BY employee_id
""")
current_df.show()

# Historical data (before raise)
print("\nHistorical data (before raise):")
historical_df = spark.sql(f"""
SELECT employee_id, name, salary as old_salary
FROM polaris.timetravel.employees VERSION AS OF {snapshot_2}
ORDER BY employee_id
""")
historical_df.show()

# Compare
print("\nComparison:")
comparison = current_df.join(historical_df, "employee_id") \
    .select(
        F.col("employee_id"),
        current_df["name"],
        F.col("old_salary"),
        F.col("current_salary"),
        (F.col("current_salary") - F.col("old_salary")).alias("increase")
    )
comparison.show()

## Part 4: Tagging Important Snapshots

### Create a Tag

In [ ]:
# Tag the current snapshot
current_snapshot = snapshots_df.collect()[-1]['snapshot_id']

print(f"Tagging snapshot {current_snapshot} as '2024-annual-review'")

# Drop tag if it exists
try:
    spark.sql("""
    ALTER TABLE polaris.timetravel.employees
    DROP TAG IF EXISTS `2024-annual-review`
    """)
except:
    pass

spark.sql(f"""
ALTER TABLE polaris.timetravel.employees
CREATE TAG `2024-annual-review` AS OF VERSION {current_snapshot}
""")

print("Tag created!")

### Query Using Tag Name

In [ ]:
# Query using tag instead of snapshot ID
print("Query using tag '2024-annual-review':")
spark.sql("""
SELECT employee_id, name, salary
FROM polaris.timetravel.employees VERSION AS OF '2024-annual-review'
ORDER BY employee_id
""").show()

### Tag an Earlier Snapshot

In [ ]:
# Tag the snapshot before salary increases
print(f"Tagging snapshot {snapshot_2} as 'pre-salary-adjustment'")

# Drop tag if it exists
try:
    spark.sql("""
    ALTER TABLE polaris.timetravel.employees
    DROP TAG IF EXISTS `pre-salary-adjustment`
    """)
except:
    pass

spark.sql(f"""
ALTER TABLE polaris.timetravel.employees
CREATE TAG `pre-salary-adjustment` AS OF VERSION {snapshot_2}
""")

print("Tag created!")

In [ ]:
# View all refs (tags and branches)
print("All refs:")
spark.sql("""
SELECT name, type, snapshot_id
FROM polaris.timetravel.employees.refs
""").show(truncate=False)

## Part 5: Make a Mistake and Rollback

### Make an Intentional Mistake

In [ ]:
# Save current snapshot ID before the mistake
good_snapshot = snapshots_df.collect()[-1]['snapshot_id']
print(f"Good snapshot ID (before mistake): {good_snapshot}")

# Oops! Accidentally delete all Engineering employees
spark.sql("""
DELETE FROM polaris.timetravel.employees
WHERE department = 'Engineering'
""")

print("\nMistake made: Deleted all Engineering employees!")

In [ ]:
# Verify the problem
print("After the mistake:")
spark.sql("""
SELECT * FROM polaris.timetravel.employees
ORDER BY employee_id
""").show()

### Rollback to Previous State

In [ ]:
# Rollback using the snapshot ID
print(f"Rolling back to snapshot {good_snapshot}...")

spark.sql(f"""
CALL polaris.system.rollback_to_snapshot(
    table => 'timetravel.employees',
    snapshot_id => {good_snapshot}
)
""")

print("Rollback complete!")

In [ ]:
# Verify recovery
print("After rollback (Engineering employees restored):")
spark.sql("""
SELECT * FROM polaris.timetravel.employees
ORDER BY employee_id
""").show()

### Alternative: Rollback Using Tag

In [ ]:
# We can also rollback using tag names
# Get snapshot ID from tag
tag_snapshot = spark.sql("""
SELECT snapshot_id FROM polaris.timetravel.employees.refs
WHERE name = '2024-annual-review'
""").collect()[0]['snapshot_id']

print(f"Tag '2024-annual-review' points to snapshot: {tag_snapshot}")
print("(We could rollback to this tag if needed)")

## Part 6: Audit Trail with Snapshots

In [ ]:
# View complete history
print("Complete snapshot history:")
spark.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation,
    summary['added-records'] as added,
    summary['deleted-records'] as deleted,
    summary['changed-partition-count'] as partitions_changed
FROM polaris.timetravel.employees.snapshots
ORDER BY committed_at
""").show(truncate=False)

## Part 7: Time-Based Queries

### Query by Timestamp

In [ ]:
# Get a timestamp between snapshots
snapshot_times = spark.sql("""
SELECT committed_at
FROM polaris.timetravel.employees.snapshots
ORDER BY committed_at
""").collect()

# Use timestamp after snapshot 1 but before snapshot 2
time_after_first = snapshot_times[0]['committed_at']
print(f"Querying as of timestamp: {time_after_first}")

print("\nData at that time (only 3 employees):")
spark.sql(f"""
SELECT * FROM polaris.timetravel.employees 
TIMESTAMP AS OF '{time_after_first}'
ORDER BY employee_id
""").show()

## Part 8: Data Archaeology Example

Use time travel to understand how data changed over time.

In [ ]:
# Track how Alice's salary changed
print("Alice Johnson's salary history:")

for i, row in enumerate(snapshots_df.collect()):
    snapshot_id = row['snapshot_id']
    timestamp = row['committed_at']
    
    result = spark.sql(f"""
    SELECT salary
    FROM polaris.timetravel.employees VERSION AS OF {snapshot_id}
    WHERE name = 'Alice Johnson'
    """)
    
    rows = result.collect()
    if rows:
        salary = rows[0]['salary']
        print(f"  Snapshot {i+1} ({timestamp}): ${salary:,.2f}")
    else:
        print(f"  Snapshot {i+1} ({timestamp}): Not yet hired")

## Key Takeaways

### Snapshots
1. **Every write creates a snapshot**: Immutable record of table state
2. **Snapshots enable time travel**: Query any historical version
3. **Snapshots enable rollback**: Undo mistakes safely
4. **Snapshots have metadata**: Operation type, timestamps, record counts

### Time Travel
1. **VERSION AS OF**: Query by snapshot ID
2. **TIMESTAMP AS OF**: Query by timestamp
3. **Tag-based queries**: Use meaningful names instead of IDs
4. **No data rewriting**: Historical versions coexist with current

### Tags
1. **Memorable names**: Better than remembering snapshot IDs
2. **Mark milestones**: End-of-quarter, pre-migration, etc.
3. **Easy reference**: Use in queries and rollback operations
4. **Audit compliance**: Required for some regulatory requirements

### Rollback
1. **Instant recovery**: Revert to any previous snapshot
2. **No data loss**: Previous snapshots still exist
3. **Disaster recovery**: Essential for production systems
4. **Metadata operation**: Very fast, no data rewriting

## Real-World Use Cases

- **Auditing**: See exactly what data looked like at any point
- **Debugging**: Identify when bad data was introduced
- **Compliance**: Prove table state for regulatory requirements
- **Recovery**: Rollback from bad ETL jobs or pipeline bugs
- **Testing**: Compare query results across versions
- **Reproducibility**: Re-run analyses on historical data

## Challenge Exercise

1. Create a table and build 5+ snapshots with different operations
2. Tag specific snapshots with meaningful names
3. Query each historical version and compare results
4. Intentionally corrupt the data, then rollback
5. Use snapshots metadata to build an audit trail

## Cleanup

In [ ]:
# Optional: Drop table to start fresh
# spark.sql("DROP TABLE IF EXISTS polaris.timetravel.employees")
# print("Table dropped!")